# Interpolation
This notebook runs a linear interpolation of SLA data in the month of Jan 2020.

In [1]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/home/mhen/.local/share/fonts/IBMPlexSerif-Regular.ttf"
font_prop = fm.FontProperties(fname=font_path)

plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["font.sans-serif"] = [font_prop.get_name()]

from GPSat.utils import WGS84toEASE2
from GPSat.local_experts import LocalExpertOI, get_results_from_h5file
from GPSat.postprocessing import smooth_hyperparameters, glue_local_predictions_1d

2026-05-18 19:07:26.563042: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 19:07:26.595744: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-18 19:07:26.595778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-18 19:07:26.597075: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-18 19:07:26.603053: I tensorflow/core/platform/cpu_feature_guar

In [2]:
data_df = pd.read_csv('/home/mhen/geol0069_data/jan20.csv')

In [3]:
#keep data from january

data_df = data_df.loc[data_df['date_string'].str.contains('2020-01', na=False)]

In [4]:
data_df.head()

,x,y,lon,lat,t,z,track,date_string,satellite,lead_mask,dist_along_track,z_track_avg
2573650,-585146.902831,-432100.026835,-53.556094,83.483790,18262.0,NaN,1243,2020-01-01,cs2,0.0,0.000000,0.040157
2573651,-584280.987782,-430861.767792,-53.594113,83.496633,18262.0,-0.034818,1243,2020-01-01,cs2,1.0,1502.990248,0.040157
2573652,-584107.820675,-430614.140368,-53.601734,83.499201,18262.0,-0.018846,1243,2020-01-01,cs2,1.0,1803.558120,0.040157
2573653,-583934.553028,-430366.439101,-53.609361,83.501770,18262.0,NaN,1243,2020-01-01,cs2,0.0,2104.243152,0.040157
2573654,-583588.157115,-429871.156267,-53.624632,83.506906,18262.0,NaN,1243,2020-01-01,cs2,0.0,2705.434872,0.040157


In [5]:
test_track = data_df.loc[data_df['track'] == 1257]
test_track.head()

,x,y,lon,lat,t,z,track,date_string,satellite,lead_mask,dist_along_track,z_track_avg
2601689,-638579.271484,2.242111e+06,-164.10243,69.001559,18262.0,-0.746479,1257,2020-01-01,cs2,1.0,0.000000,-0.026275
2601690,-638462.214874,2.241838e+06,-164.10336,69.004244,18262.0,-0.798415,1257,2020-01-01,cs2,1.0,300.176217,-0.026275
2601691,-638111.488080,2.241019e+06,-164.10614,69.012299,18262.0,-0.665761,1257,2020-01-01,cs2,1.0,1200.652652,-0.026275
2601692,-637409.906412,2.239382e+06,-164.11171,69.028408,18262.0,-0.714593,1257,2020-01-01,cs2,1.0,3001.529271,-0.026275
2601693,-637292.936222,2.239109e+06,-164.11264,69.031093,18262.0,-0.669882,1257,2020-01-01,cs2,1.0,3301.699913,-0.026275


In [9]:
def random_holdout_test(data_df, holdout_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)

    #separate into tracks and iterate over each
    tracks = data_df['track'].unique()
    
    results_df = pd.DataFrame()
    results_metrics_df = pd.DataFrame()
    for i, track in enumerate(tracks):
        print(f'Interpolating track {track}, {i+1} of {len(tracks)}...')
        track_df = data_df.loc[data_df['track']==track]
        #locate leads
        leads_df = track_df.loc[track_df['lead_mask']==1.0]
        #create test (holdout) indices of length 1 or of holdout fraction, whichever is larger
        n_test = max(1, int(len(leads_df) * holdout_fraction))

        #exclude tracks with insufficient leads
        if len(leads_df) < 5:
            print(f"Insufficient leads ({len(leads_df)}) for random holdout, skipping...")
            continue

        #find lead indices and randomly select 80% for training
        leads_idx = leads_df.index
        test_lead_idx = rng.choice(leads_df.index, size=n_test, replace=False)
        train_lead_idx = np.array(leads_df.index.difference(test_lead_idx))

        # Compute distances before interpolation - nearest distance to a train lead of each test lead
        lead_D = leads_df.loc[leads_idx, 'dist_along_track'].to_numpy()
        train_D = leads_df.loc[train_lead_idx, 'dist_along_track'].to_numpy()
        test_D = leads_df.loc[test_lead_idx, 'dist_along_track'].to_numpy()
        if train_D.size > 0 and test_D.size > 0:
            nearest_dists_km = np.min(np.abs(test_D[:, None] - train_D[None, :]), axis=1) / 1000.0
        else:
            nearest_dists_km = np.array([])

        #run linear interpolation
        result_track_lin = linear_interpolation(track_df, train_lead_idx)
        result_track_gpsat = gpsat_interpolation(track_df, train_lead_idx)
        print(f'lin:')
        print(result_track_lin.head())
        print('gpsat:')
        print(result_track_gpsat.head())
        result_track = result_track_lin.join(result_track_gpsat[['f_gpsat','f_var_gpsat']])

        #find test values - withheld leads
        linear_preds = result_track_lin.loc[test_lead_idx,'f_lin'].to_numpy()
        gpsat_preds = result_track_gpsat.loc[test_lead_idx, 'f_gpsat'].to_numpy()
        gpsat_var = result_track_gpsat.loc[test_lead_idx,'f_var_gpsat'].to_numpy()
        targets = result_track_lin.loc[test_lead_idx, 'z'].to_numpy()

        print(f"Test NaNs — Linear: {np.sum(~np.isfinite(linear_preds))}, GPSat: {np.sum(~np.isfinite(gpsat_preds))}")
        both_valid = (np.isfinite(linear_preds) & np.isfinite(gpsat_preds) & np.isfinite(targets))
        if np.sum(both_valid) == 0:
            print(f"FAILED: No valid predictions from both methods on random holdout")
            continue

        #find metrics for the prediction
        lin_res = linear_preds[both_valid] - targets[both_valid]
        gp_res = gpsat_preds[both_valid] - targets[both_valid]

        def _r2(a, b):
            return np.corrcoef(a, b)[0,1]**2 if len(a) > 1 else np.nan
        
        # Calculate GPSat uncertainty at test leads for calibration check
        if gpsat_var is not None:
            gpsat_std = np.sqrt(gpsat_var[both_valid])
            avg_gpsat_uncertainty = float(np.mean(gpsat_std))
        else:
            avg_gpsat_uncertainty = None
        
        # Distance and lead spacing stats
        dist_mean = float(np.nanmean(nearest_dists_km)) if nearest_dists_km.size else np.nan
        dist_median = float(np.nanmedian(nearest_dists_km)) if nearest_dists_km.size else np.nan
        dist_max = float(np.nanmax(nearest_dists_km)) if nearest_dists_km.size else np.nan
        dist_min = float(np.nanmin(nearest_dists_km)) if nearest_dists_km.size else np.nan
        spacing_km = np.diff(lead_D) / 1000.0 if lead_D.size > 1 else np.array([])
        sp_med = float(np.nanmedian(spacing_km)) if spacing_km.size else np.nan
        sp_p90 = float(np.nanpercentile(spacing_km, 90)) if spacing_km.size else np.nan
        sp_max = float(np.nanmax(spacing_km)) if spacing_km.size else np.nan

        #add metrics to dataframe
        metrics_dict = {
            'track' : track,
            'n_predictions': int(np.sum(both_valid)),
            'n_train': int(len(train_lead_idx)),
            'n_test': int(len(test_lead_idx)),
            'nearest_train_km': {
                'mean': dist_mean, 'median': dist_median, 'max': dist_max, 'min': dist_min
            },
            'lead_spacing_km': {
                'median': sp_med, 'p90': sp_p90, 'max': sp_max
            },
            'linear': {
                'rmse': float(np.sqrt(np.mean(lin_res**2))),
                'bias': float(np.mean(lin_res)),
                'mae': float(np.mean(np.abs(lin_res))),
                'r2': float(_r2(linear_preds[both_valid], targets[both_valid]))
            },
            'gpsat_along': {
                'rmse': float(np.sqrt(np.mean(gp_res**2))),
                'bias': float(np.mean(gp_res)),
                'mae': float(np.mean(np.abs(gp_res))),
                'r2': float(_r2(gpsat_preds[both_valid], targets[both_valid])),
                'avg_uncertainty': avg_gpsat_uncertainty
                },
        }
        track_metrics_df = pd.DataFrame(metrics_dict)
        results_metrics_df = pd.concat([results_metrics_df, track_metrics_df])
        results_df = pd.concat([results_df, result_track])
        print(f'Track interpolated!')
    print('All tracks interpolated')

    return results_df, results_metrics_df


def linear_interpolation(track_df, train_lead_idx, filter_width_km=100):
    obs_df = track_df['z']
    train_leads_df = track_df.loc[train_lead_idx]

    # Filter leads by height
    height_valid = (train_leads_df['z']>= -0.5) & (train_leads_df['z'] <= 0.5)
    train_leads_df = train_leads_df.loc[height_valid]

    #terminate if too short for linear interpolation
    if len(train_leads_df) < 2:
        print('Too few leads for linear interpolation, returning NaN')
        obs_df['f_lin'] = np.nan
        track_df = pd.merge(track_df, obs_df['f_lin'], left_index=True, right_index=True)
        
        return track_df
    
    # apply 100 km box filter to smooth lead observations
    smoothed_obs_df = obs_df.copy()

    # For each lead point, smooth with nearby leads within filter_width_km
    for idx, lead in train_leads_df.iterrows():
        lead_distances = np.abs(train_leads_df['dist_along_track'] - train_leads_df['dist_along_track'].loc[idx])
        within_window = lead_distances <= (filter_width_km * 1000 / 2)  # ±50 km    
        if np.sum(within_window) > 0:
            smoothed_obs_df.loc[idx] = (train_leads_df['z'][within_window]).mean()

    # linear interpolation between smoothed lead values - covers all obs points
    obs_df = obs_df.to_frame()
    obs_df['f_lin'] = np.nan
        
    for i0, i1 in zip(train_leads_df.index[:-1], train_leads_df.index[1:]):
        v0, v1 = smoothed_obs_df.loc[i0], smoothed_obs_df.loc[i1]
        span = i1 - i0
        if span <= 0:
            obs_df['f_lin'].loc[i0] = v0
            continue
        ii = np.arange(i0, i1 + 1)
        obs_df['f_lin'].loc[ii] = v0 + (v1 - v0) * (ii - i0) / span

    # flat extrapolation beyond the first/last training lead
    first_idx, last_idx = train_leads_df.index[0], train_leads_df.index[-1]
    obs_df.loc[:first_idx, 'f_lin'] = smoothed_obs_df.loc[first_idx]
    obs_df.loc[last_idx:, 'f_lin'] = smoothed_obs_df.loc[last_idx]

    track_df = pd.merge(track_df, obs_df['f_lin'], left_index=True, right_index=True)
    track_df.head()

    return track_df

def gpsat_interpolation(track_df, train_lead_idx):
    
    obs_df = track_df['z']
    train_leads_df = track_df.loc[train_lead_idx]

    # Filter leads by height
    height_valid = (train_leads_df['z']>= -0.5) & (train_leads_df['z'] <= 0.5)
    train_leads_df = train_leads_df.loc[height_valid]

    #if too short, terminate and return NaN
    if len(train_leads_df)<3:
        print('Too few leads for GPSat interpolation, returning NaN')
        track_df['f_gpsat'] = np.nan
        
        return track_df
    
    #select experts
    experts_df = select_experts(train_leads_df)

    if len(experts_df) < 2:
        print('Too few experts for GPSat, returning NaN')
        track_df['f_gpsat'] = np.nan
        
        return track_df
    
    #run gpsat interpolation
    track = track_df['track'].iloc[0]

    store_path = f'/home/mhen/geol0069_data/gpsat/jan20/{track}.h5'

    if os.path.exists(store_path):
        os.remove(store_path)
    
    ssha, ssha_var = run_gpsat(track_df, train_leads_df, experts_df, store_path)

    track_df['f_gpsat'] = ssha
    track_df['f_var_gpsat'] = ssha_var

    return track_df

def select_experts(leads_df, spacing=25e3, max_experts=40):
    try:
        dist_along_track = leads_df['dist_along_track']
    except KeyError as ke:
        print(f'{ke}: Make sure distance along track is calculated')
        return
    
    grid = np.arange(0, dist_along_track.iloc[-1], spacing)
    selected = []
    used = set()

    for target in grid:
        if len(selected) >= max_experts:
            print('Expert limit reached.')
            break
        #subtract each target dist from the dist of each lead, and so find
        #the closest lead to target from the smallest value
        idx = (dist_along_track-target).abs().idxmin()
        #select lead and add to selected and used
        if idx not in used:
            selected.append(idx)
            used.add(idx)
    
    if len(selected) < 3:
        #if we have too few experts
        #find the unselected leads
        remaining_idx = leads_df.index.difference(selected)
        #find no. experts to add
        n_to_add = max_experts-len(remaining_idx)
        #add indices evenly spaced from remaining unselected indices
        step = max(1, len(remaining_idx) // n_to_add)
        idx_added = remaining_idx[::step][:n_to_add]

        selected.extend(list(idx_added))

    experts_df = leads_df.loc[selected]

    return experts_df

def run_gpsat(track_df, leads_df, experts_df, store_path):

    # Notebook-style config: 3D coords, 400km, 100-200km lengthscales
    coords_col = ["x", "y", "t"]
    local_select = [
        {"col": ["x", "y"], "comp": "<=", "val": 400_000},
        {"col": "t", "comp": "<=", "val": 1.0},
        {"col": "t", "comp": ">=", "val": -1.0}
    ]
    data_config = {
        "data_source": leads_df,
        "obs_col": "z",
        "coords_col": coords_col,
        "local_select": local_select,
        "global_select": [{"col": "lat", "comp": ">=", "val": 45}]
    }
    expert_config = {"source" : experts_df}
    pred_config = {"method": "from_dataframe", "df": track_df,
                   "max_dist" : 400_000}
    # Notebook-style: 100-200km lengthscales for 3D
    length_low = [100_000, 100_000, 1e-08]
    length_high = [200_000, 200_000, 4]
    coords_scale = [10_000, 10_000, 1]

    model_config = {
    "oi_model": "GPflowGPRModel",
    "init_params": {"coords_scale": coords_scale, "minibatch_size": 100},
    "constraints": {
        "lengthscales": {"low": length_low, "high": length_high},
        "likelihood_variance": {"low": 0.001, "high": 10}  # Lower noise floor for better endpoint adherence
        }
    }
    # Full mode: enable smoothing (3D coords have x/y)
    locexp = LocalExpertOI(expert_config, data_config, model_config, pred_config)
    locexp.run(store_path=store_path, optimise=True)
    dfs, _ = get_results_from_h5file(store_path)
    run_details = dfs['run_details']
    smooth_hyperparameters(
        result_file=store_path,
        params_to_smooth=["lengthscales", "kernel_variance", "likelihood_variance"],
        smooth_config_dict={
            "lengthscales": dict(l_x=200_000, l_y=200_000, max=12),
            "likelihood_variance": dict(l_x=200_000, l_y=200_000),
            "kernel_variance": dict(l_x=200_000, l_y=200_000, max=0.1)
        },
        save_config_file=False
    )
    model_config['load_params'] = {"file": store_path, "table_suffix": "_SMOOTHED"}
    locexp = LocalExpertOI(expert_config, data_config, model_config, pred_config)
    locexp.run(store_path=store_path, optimise=False, predict=True, table_suffix='_SMOOTHED')

    # Extract results
    dfs, _ = get_results_from_h5file(store_path)
    preds = dfs["preds"]

    # Glue
    preds['xprt_locs'] = np.sqrt((preds['x'] - preds['x'].iloc[0])**2 + (preds['y'] - preds['y'].iloc[0])**2)
    preds['pred_locs'] = np.sqrt((preds['pred_loc_x'] - preds['pred_loc_x'].iloc[0])**2 + 
                                    (preds['pred_loc_y'] - preds['pred_loc_y'].iloc[0])**2)
    glue_max = 400_000  # Full mode uses 400km like notebook

    glued = glue_local_predictions_1d(preds, "pred_locs", "xprt_locs", ["f*", "f*_var"], glue_max)
    ssha = glued['f*'].values
    ssha_var = glued['f*_var'].values if 'f*_var' in glued.columns else None

    # Handle length mismatch
    if len(ssha) < len(track_df):
        full = np.full(len(track_df), np.nan)
        full_var = np.full(len(track_df), np.nan)
        full[:len(ssha)] = ssha
        full_var[:len(ssha_var)] = ssha

        return full, full_var
    
    return ssha[:len(track_df)], ssha_var[:len(track_df)]

In [10]:
tracks = data_df['track'].unique()

tracks = tracks[:15]

downsized_df = data_df.loc[data_df['track'].isin(tracks)]

In [11]:
linear_df, linear_metrics_df = random_holdout_test(downsized_df)

Interpolating track 1243, 1 of 15...
Expert limit reached.
'data_select': 0.000 seconds
'load': 0.000 seconds
in json_serializable - key: 'data_source' has value DataFrame/Series, but is too long: 1290 >  100
storing as str
in json_serializable - key: 'df' has value DataFrame/Series, but is too long: 5205 >  100
storing as str
---------
storing expert locations in 'expert_locs' table
exception occurred: 'No object named expert_locs in the file'
will now close object

---------
dropping expert locations that already exists in 'run_details' table
exception occurred: 'No object named run_details in the file'
will now close object

--------------------------------------------------
1 / 40
current local expert:
                     x              y        lon        lat        t  \
2573651 -584280.987782 -430861.767792 -53.594113  83.496633  18262.0   

                z  track date_string satellite  lead_mask  dist_along_track  \
2573651 -0.034818   1243  2020-01-01       cs2        1.0   

2026-05-18 19:13:11.697374: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 20872 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:42:00.0, compute capability: 8.9
2026-05-18 19:13:11.698798: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:1 with 20872 MB memory:  -> device: 1, name: NVIDIA L4, pci bus id: 0000:c4:00.0, compute capability: 8.9


**********
optimization failed!
'optimise_parameters': 0.294 seconds
'get_parameters': 0.003 seconds
parameters:
lengthscales: array([10.00997233, 10.00994317,  0.99822433]) 
kernel_variance: 0.0013667877537881785
likelihood_variance: 0.004187448605488844
'predict': 0.030 seconds
total run time : 0.72 seconds
--------------------------------------------------
2 / 40
current local expert:
                     x              y        lon        lat        t        z  \
2573685 -570418.599441 -411052.893457 -54.222867  83.701745  18262.0 -0.00557   

         track date_string satellite  lead_mask  dist_along_track  z_track_avg  
2573685   1243  2020-01-01       cs2        1.0      25551.212003     0.040157  
'local_data_select': 0.001 seconds
number obs: 124
found GPU
setting lengthscales to: [1. 1. 1.]
'__init__': 0.049 seconds
'set_lengthscales_constraints': 0.006 seconds
'set_likelihood_variance_constraints': 0.010 seconds


2026-05-18 19:13:12.419808: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 20872 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:42:00.0, compute capability: 8.9
2026-05-18 19:13:12.421206: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:1 with 20872 MB memory:  -> device: 1, name: NVIDIA L4, pci bus id: 0000:c4:00.0, compute capability: 8.9


'optimise_parameters': 0.283 seconds
'get_parameters': 0.003 seconds
parameters:
lengthscales: array([10.0100683 , 10.0101367 ,  1.00053476]) 
kernel_variance: 0.001264155332275684
likelihood_variance: 0.003928569402151681
'predict': 0.030 seconds
total run time : 0.71 seconds
--------------------------------------------------
3 / 40
current local expert:
                     x              y        lon        lat        t  \
2573741 -555327.315351 -389517.775035 -54.953406  83.923938  18262.0   

                z  track date_string satellite  lead_mask  dist_along_track  \
2573741  0.000517   1243  2020-01-01       cs2        1.0      51704.152824   

         z_track_avg  
2573741     0.040157  
'local_data_select': 0.001 seconds
number obs: 133
found GPU
setting lengthscales to: [1. 1. 1.]
'__init__': 0.050 seconds
'set_lengthscales_constraints': 0.006 seconds
'set_likelihood_variance_constraints': 0.010 seconds


...

2026-05-18 19:27:23.610422: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 20872 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:42:00.0, compute capability: 8.9
2026-05-18 19:27:23.612577: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:1 with 20872 MB memory:  -> device: 1, name: NVIDIA L4, pci bus id: 0000:c4:00.0, compute capability: 8.9


total run time : 1.03 seconds
--------------------------------------------------
40 / 40
current local expert:
                     x              y        lon        lat        t  \
2618775  484983.566989  397357.891614  129.32853  84.384209  18262.0   

                z  track date_string satellite  lead_mask  dist_along_track  \
2618775  0.058939   1260  2020-01-01       cs2        1.0     975011.018558   

         z_track_avg  
2618775      0.09918  
'local_data_select': 0.001 seconds
number obs: 405
found GPU
setting lengthscales to: [1. 1. 1.]
'__init__': 0.051 seconds
'_read_params_from_file': 0.044 seconds
'set_parameters': 0.005 seconds
'set_lengthscales_constraints': 0.005 seconds
'set_likelihood_variance_constraints': 0.015 seconds
*** not optimising parameters
'get_parameters': 0.003 seconds
parameters:
lengthscales: array([10.01489149, 10.01488959,  1.00156709]) 
kernel_variance: 0.006335221919568361
likelihood_variance: 0.01100000000000001
'predict': 0.036 seconds


2026-05-18 19:27:24.645993: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 20872 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:42:00.0, compute capability: 8.9
2026-05-18 19:27:24.647406: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:1 with 20872 MB memory:  -> device: 1, name: NVIDIA L4, pci bus id: 0000:c4:00.0, compute capability: 8.9


SAVING RESULTS TO TABLES:
run_details
preds
total run time : 1.17 seconds
'run': 41.451 seconds
reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
lin:
                     x             y        lon        lat        t         z  \
2617067  950053.565564  1.257429e+06  142.92703  75.851377  18262.0  0.071572   
2617068  948798.845002  1.255036e+06  142.91097  75.875434  18262.0 -0.027448   
2617069  948380.678230  1.254239e+06  142.90560  75.883453  18262.0  0.082242   
2617070  947962.365305  1.253441e+06  142.90023  75.891472  18262.0  0.012394   
2617071  947683.408832  1.252910e+06  142.89665  75.896818  18262.0  0.022885   

         track date_string satellite  lead_mask  dist_along_track  \
2617067   1260  2020-01-01       cs2        1.

/tmp/ipykernel_1171781/839927469.py:200: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track_df['f_gpsat'] = ssha
/tmp/ipykernel_1171781/839927469.py:201: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track_df['f_var_gpsat'] = ssha_var
/tmp/ipykernel_1171781/839927469.py:66: RuntimeWarning: invalid value encountered in sqrt
  gpsat_std = np.sqrt(gpsat_var[both_valid])


In [12]:
dfs, _ = get_results_from_h5file('/home/mhen/geol0069_data/gpsat/jan20/1243.h5')

run_details = dfs['run_details']

reading in results
getting all tables
merging on expert location data
table: 'oi_config' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
table: 'oi_config_SMOOTHED' does not have all coords_col: ['x', 'y', 't'] in columns, not merging on expert_locations
               x              y        t  _dim_0  num_obs  run_time  \
0 -584280.987782 -430861.767792  18262.0       0      116  0.408209   
1 -570418.599441 -411052.893457  18262.0       0      124  0.390630   
2 -555327.315351 -389517.775035  18262.0       0      133  0.403390   
3 -542825.464630 -371701.254797  18262.0       0      136  0.367528   
4 -525095.068321 -346469.863559  18262.0       0      139  0.368807   

   objective_value  parameters_optimised  optimise_success  \
0      -148.145213                  True             False   
1      -162.322020                  True              True   
2      -176.598838                  True              True   
3      -177.445625          

In [13]:
print(dfs.keys())

dict_keys(['expert_locs', 'expert_locs_SMOOTHED', 'kernel_variance', 'kernel_variance_SMOOTHED', 'lengthscales', 'lengthscales_SMOOTHED', 'likelihood_variance', 'likelihood_variance_SMOOTHED', 'oi_config', 'oi_config_SMOOTHED', 'preds', 'preds_SMOOTHED', 'run_details', 'run_details_SMOOTHED'])
